# Simulateur de trafic — modèle discret

Route **circulaire** à plusieurs voies ; chaque voiture est un agent.
Toute la logique (classes + schéma numérique) est dans `simulateur_trafic.py`.
Ce notebook sert à **définir les paramètres**, **lancer la simulation** et
**visualiser** les résultats.

**Modèle d'accélération** (voiture `i`, voiture de devant `j`) :

$$ a_i = \mu_i\,\big(\underbrace{d_i + (v_j - v_i)\,\Delta t}_{\text{écart prévu}} - d^{\text{sec}}_i\big),
\qquad v_i \leftarrow \mathrm{clip}(v_i + a_i\Delta t,\,0,\,v^{\max}_i) $$

- `mu` : coefficient de sensibilité de l'accélération
- `d_sec` : distance de sécurité
- `v_max` : vitesse maximale

**Profils** : prudent, normal, fou (ignore la voiture arrière en doublant),
camion (interdit de 3e voie). Dépassement par la **gauche** uniquement,
rabattement à **droite** dès que possible (règles européennes).

## 1. Imports

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from matplotlib.lines import Line2D
from IPython.display import HTML

from simulateur_trafic import Parametres, Simulation, construire_profils

## 2. Paramètres

On modifie ici **tous** les paramètres de la simulation. Les profils
(prudent / normal / fou / camion) sont dérivés automatiquement des valeurs de
base `mu`, `distance_securite` et `vitesse_max`.

In [ ]:
params = Parametres(
    # --- Route ---
    L=1000.0,        # longueur de la route (m)
    N=75,            # nombre de voitures
    N_voie=3,        # nombre de voies (1 = droite, ..., N_voie = gauche)

    # --- Temps ---
    dt=0.01,         # pas de temps (s)
    T=120.0,         # durée de simulation (s)  -- réduire pour des essais rapides

    # --- Physique ---
    a_max=3.0, a_min=-7.0,
    longueur_voiture=3.0,
    distance_min=1.0,        # interstice minimal en bouchon (m)

    # --- Comportement ---
    temps_reaction=0.8,      # temps de reaction du conducteur (s) ; 0 = instantane
                             # (rend les rabattements serres dangereux : freinage tardif)

    # --- Valeurs de base (profil normal) ---
    mu=0.4,                  # sensibilité de l'accélération
    distance_securite=10.0,  # distance de sécurité (m)
    vitesse_max=30.0,        # vitesse max (m/s)

    # --- Population ---
    proportions={"prudent": 0.20, "normal": 0.50, "fou": 0.20, "camion": 0.10},
    vitesse_initiale=22.0,
    nb_arret_initial=3,      # voitures à l'arrêt au départ (crée une onde de bouchon)
    graine=42,               # graine aléatoire (reproductibilité)
)

print(f"{params.nb_iterations} pas de temps a simuler")

### (optionnel) Régler finement les profils

Les profils dérivent des valeurs de base ci-dessus. Pour les ajuster un par un,
décommenter et passer `profils=...` à la `Simulation`.

In [ ]:
# profils = construire_profils(params)
# profils["fou"].vitesse_max = 40.0
# profils["camion"].voie_max = 2      # voie la plus a gauche autorisee
# sim = Simulation(params, profils=profils)

## 3. Lancer la simulation

In [ ]:
sim = Simulation(params)
sim.simuler(verbeux=True)

print("Termine. Formes des historiques :", sim.X.shape)
print("Vitesse moyenne finale :", round(float(sim.V[-1].mean()), 2), "m/s")

## 4. Analyse — vitesse moyenne par profil

Permet de vérifier l'écart de comportement : le fou roule plus vite, le camion
plus lentement, etc.

In [ ]:
plt.figure(figsize=(10, 5))
for nom in params.proportions:
    indices = [i for i, prof in enumerate(sim.profils_voitures) if prof == nom]
    if indices:
        v_moy = sim.V[:, indices].mean(axis=1)
        plt.plot(sim.temps, v_moy, label=nom, color=sim.profils[nom].couleur)
plt.xlabel("Temps (s)")
plt.ylabel("Vitesse moyenne (m/s)")
plt.title("Vitesse moyenne selon le profil de conducteur")
plt.legend(); plt.grid(True); plt.show()

## 5. Animation — route circulaire (matplotlib, dans le notebook)

Voie 1 = cercle **extérieur** (droite), voie `N_voie` = cercle **intérieur** (gauche).

Les changements de voie sont **progressifs** : on dessine la position latérale
continue `sim.YLAT` (et non la voie entière `sim.VOIE`), donc une voiture *glisse*
d'une voie à l'autre au lieu de sauter. On saute des images (`pas_anim`) pour
alléger l'animation.

> Pour une version **temps réel, plus fluide et interactive**, voir la section 6
> (fenêtre **pygame**).

In [ ]:
plt.rcParams["animation.embed_limit"] = 50  # Mo

# Rayon de chaque voie : voie 1 a l'exterieur, voie N_voie a l'interieur.
rayons_voies = np.linspace(1.25, 0.95, params.N_voie)

def rayon_continu(lat):
    """Rayon (continu) correspondant a une position laterale fractionnaire."""
    if params.N_voie == 1:
        return np.full_like(np.asarray(lat, dtype=float), rayons_voies[0])
    return np.interp(lat, np.arange(1, params.N_voie + 1), rayons_voies)

fig, ax = plt.subplots(figsize=(6, 6), dpi=90)
theta = np.linspace(0, 2 * np.pi, 400)
for r in rayons_voies:
    ax.plot(r * np.cos(theta), r * np.sin(theta), "--", color="0.6", lw=0.8, alpha=0.5)

def positions_frame(frame):
    angles = 2 * np.pi * (sim.X[frame] % params.L) / params.L
    r = rayon_continu(sim.YLAT[frame])          # YLAT continu => glissement lisse
    return r * np.cos(angles), r * np.sin(angles), r

# Taille PHYSIQUE des marqueurs : leur diametre a l'ecran vaut la longueur reelle
# d'une voiture. Sinon les marqueurs (taille fixe) se chevauchent dans les bouchons.
# La voie interieure (rayon plus petit) donne des voitures plus petites.
# Taille des voitures a l'ecran (x longueur reelle). Augmenter pour des
# marqueurs plus gros ; au-dela de ~2 ils se chevauchent dans les bouchons.
ZOOM = 1.8
LONG_VOIT_M = ZOOM * params.longueur_voiture    # longueur de rendu (m)
def tailles(r):
    diam_data = LONG_VOIT_M * (2 * np.pi * r) / params.L         # arc le long de l'anneau
    p0 = ax.transData.transform((0, 0)); p1 = ax.transData.transform((1, 0))
    pts_par_unite = np.hypot(*(p1 - p0)) * 72 / fig.dpi          # points par unite 'data'
    return (diam_data * pts_par_unite) ** 2                      # scatter s = (diam en points)^2

x0, y0, r0 = positions_frame(0)
points = ax.scatter(x0, y0, c=sim.couleurs, zorder=3,
                    edgecolors="white", linewidths=0.4)

ax.set_aspect("equal")
ax.set_xlim(-1.4, 1.4); ax.set_ylim(-1.4, 1.4)
ax.set_xticks([]); ax.set_yticks([])
fig.canvas.draw()                 # fige la transformation data->ecran avant le calcul des tailles
points.set_sizes(tailles(r0))
legende = [Line2D([0], [0], marker="o", linestyle="", label=nom,
                  markerfacecolor=sim.profils[nom].couleur, markeredgecolor="none",
                  markersize=8) for nom in params.proportions]
ax.legend(handles=legende, loc="upper right", fontsize=8)

pas_anim = 20  # 1 image affichee tous les 20 pas
frames = range(0, len(sim.temps), pas_anim)

def update(frame):
    xs, ys, r = positions_frame(frame)
    points.set_offsets(np.column_stack([xs, ys]))
    points.set_sizes(tailles(r))
    ax.set_title(f"Route circulaire a {params.N_voie} voies - t = {sim.temps[frame]:.1f} s")
    return points,

anim = FuncAnimation(fig, update, frames=frames, interval=40, blit=False)
plt.close(fig)
HTML(anim.to_jshtml())

## 6. Animation pygame — fenêtre temps réel, fluide et interactive

Ouvre une **fenêtre séparée** (indépendante du notebook) qui rejoue la simulation
en temps réel, avec interpolation entre les pas et changements de voie
progressifs. Le rendu est bien plus fluide que l'animation embarquée ci-dessus.

**Commandes :** `Espace` pause · `↑`/`↓` vitesse de lecture · `←`/`→` ±2 s ·
`R` début · `Échap` quitter.

La fenêtre relance sa propre simulation avec les `params` ci-dessus. On peut
aussi la lancer depuis un terminal : `python3 animation_pygame.py --duree 60`.

In [ ]:
import subprocess

# Utilise le python3 systeme (pas le kernel Jupyter) pour avoir acces a pygame.
PYTHON = "/usr/bin/python3"

# Lance la fenetre pygame dans un processus separe (ne bloque pas le kernel).
cmd = [PYTHON, "animation_pygame.py",
       "--duree", str(int(params.T)),
       "--voitures", str(params.N),
       "--voies", str(params.N_voie),
       "--tcv", str(params.tps_changement_voie),
       "--treaction", str(params.temps_reaction)]
if params.graine is not None:
    cmd += ["--graine", str(params.graine)]

subprocess.Popen(cmd)
print("Fenetre pygame lancee — cherchez-la parmi vos fenetres.")
print(" ".join(cmd))